# 토크나이저 학습 + 검증 (Colab 단독 실행)

이 노트북만 Colab에 업로드하고 위에서부터 실행하세요. 저장소 clone이나 별도 Python 파일은 필요하지 않습니다. CPU 런타임으로 실행할 수 있습니다.

- 입력: Drive의 `korean_sllm_data/pair_train.jsonl`, `korean_sllm_data/pretrain/pretrain_train.json`. 각 파일 대신 `<파일명>.tar.xz`를 업로드해도 됩니다. 위치가 다르면 준비 셀의 경로를 수정하세요.
- JSONL은 user/assistant 필드, pretrain JSON은 문자열 배열을 사용합니다. 대용량 JSON은 스트리밍으로 읽습니다.
- 코퍼스와 압축 해제 파일은 Colab 로컬 디스크에 저장합니다. 디스크 여유 공간과 RAM이 필요합니다. SentencePiece는 최대 2,000,000줄을 표본 추출합니다.
- Unigram / NFKC / vocab 32768, 특수 토큰 ID는 기존 모델과 동일합니다.
- 산출물은 Drive의 `korean_sllm_data/tokenizer/spm.model`, `spm.vocab`에 저장합니다. 사전학습에 사용하려면 **새 학습 시작 전** `spm.model`을 `korean_sllm_data/pretrain/spm.model`로 복사하세요. 진행 중인 학습의 토크나이저는 바꾸지 마세요.
- 기존 모델 검증만 하려면 준비 셀까지 실행하고 코퍼스 추출·학습 셀을 건너뛰세요. 이 경우에도 대화 통계에는 pair 데이터가 필요합니다.


In [ ]:
# 0) 패키지 설치 및 Drive 연결
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'pandas', 'ijson>=3.2.0'], check=True)
from pathlib import Path
import json, tarfile, unicodedata, os, shutil
import pandas as pd
import sentencepiece as spm
import ijson
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/korean_sllm_data')
PAIR_SOURCE = DATA_DIR / 'pair_train.jsonl'
PRETRAIN_SOURCE = DATA_DIR / 'pretrain' / 'pretrain_train.json'
OUTPUT_DIR = DATA_DIR / 'tokenizer'
ROOT = Path('/content/tokenizer_work')
OUT_DIR = ROOT / 'tokenizer'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORPUS = ROOT / 'tokenizer_corpus.txt'
MODEL_PATH = OUT_DIR / 'spm.model'
if (OUTPUT_DIR / 'spm.model').is_file():
    shutil.copy2(OUTPUT_DIR / 'spm.model', MODEL_PATH)

PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
SPECIAL_TURN_TOKENS = ['<start_of_turn>', '<end_of_turn>']
NUM_CPUS = os.cpu_count() or 1


def ensure_input(source):
    if source.is_file():
        print('입력:', source)
        return source
    archive = source.with_name(source.name + '.tar.xz')
    if not archive.is_file():
        raise FileNotFoundError(f'{source} 또는 {archive}를 Drive에 업로드하세요.')
    target = ROOT / source.name
    print('압축 해제:', archive, flush=True)
    with tarfile.open(archive, 'r:xz') as tar:
        matches = [m for m in tar.getmembers() if m.isfile() and Path(m.name).name == source.name]
        if len(matches) != 1:
            raise ValueError(f'{archive}: {source.name} 파일이 정확히 하나 있어야 합니다.')
        with tar.extractfile(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    return target


def iter_texts(source):
    with Path(source).open('rb') as stream:
        events = ijson.parse(stream)
        if next(events, None) != ('', 'start_array', None):
            raise ValueError(f'{source}: JSON 문자열 배열이 필요합니다.')
        for prefix, event, value in events:
            if prefix == '' and event == 'end_array':
                if next(events, None) is not None:
                    raise ValueError('배열 뒤에 데이터가 있습니다.')
                return
            if prefix != 'item' or event != 'string':
                raise ValueError('모든 배열 원소는 문자열이어야 합니다.')
            yield value
        raise ValueError('JSON 배열이 완성되지 않았습니다.')


def extract_corpus(jsonl_path: Path, corpus_path: Path) -> int:
    # 순수 I/O + json 파싱이라 단일 코어로도 수십 초면 끝난다.
    # (multiprocessing 으로 나눠 봤지만 문자열 pickling 비용이 더 커서 오히려 느렸음)
    n = 0
    with jsonl_path.open(encoding="utf-8") as src, corpus_path.open("w", encoding="utf-8") as out:
        for line in src:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            for key in ("user", "assistant"):
                text = obj.get(key, "").strip()
                if text:
                    out.write(text.replace("\n", " ") + "\n")
                    n += 1
    return n

def train(corpus_path: Path, vocab_size: int, num_threads: int = NUM_CPUS) -> Path:
    spm.SentencePieceTrainer.train(
        num_threads=num_threads,  # EM 학습 단계를 모든 CPU 코어에 분산
        input=str(corpus_path),
        model_prefix=str(OUT_DIR / "spm"),
        model_type="unigram",
        vocab_size=vocab_size,
        normalization_rule_name="nfkc",
        # 0.9995 에서는 커버리지 밖 희귀 문자(한자·이모지·희귀 음절)가 vocab 을 늘려도
        # byte-fallback 으로 남았다(토큰의 2.7%). docs/model_config_review.md §4 참조.
        character_coverage=0.9999,
        byte_fallback=True,
        split_digits=True,
        remove_extra_whitespaces=False,
        allow_whitespace_only_pieces=True,
        max_sentence_length=32768,
        input_sentence_size=2_000_000,
        shuffle_input_sentence=True,
        pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID,
        pad_piece="<pad>", bos_piece="<bos>", eos_piece="<eos>", unk_piece="<unk>",
        user_defined_symbols=SPECIAL_TURN_TOKENS,
    )
    return OUT_DIR / "spm.model"

def encode_sample(sp: spm.SentencePieceProcessor, user: str, assistant: str) -> tuple[list[int], list[int]]:
    prompt_ids = [sp.bos_id()] + sp.encode(f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n")
    answer_ids = sp.encode(f"{assistant}<end_of_turn>") + [sp.eos_id()]
    ids = prompt_ids + answer_ids
    mask = [0] * len(prompt_ids) + [1] * len(answer_ids)
    return ids, mask


In [ ]:
# 1) 두 데이터셋을 한 코퍼스로 추출 (대용량 파일이므로 시간이 걸릴 수 있음)
#    JSON 배열을 통째로 메모리에 올리지 않고 노트북에 포함된 스트리밍 파서를 사용한다.
def extract_training_corpus(pair_path, pretrain_path, corpus_path):
    tmp = corpus_path.with_suffix('.tmp')
    try:
        n_pair = extract_corpus(pair_path, tmp)
        print(f'pair: {n_pair:,}줄', flush=True)
        n_pretrain = 0
        with tmp.open('a', encoding='utf-8') as out:
            for text in iter_texts(pretrain_path):
                text = text.strip()
                if text:
                    out.write(text.replace('\r', ' ').replace('\n', ' ') + '\n')
                    n_pretrain += 1
                    if n_pretrain % 100_000 == 0:
                        print(f'pretrain: {n_pretrain:,}줄 처리 중', flush=True)
        if not n_pair or not n_pretrain:
            raise ValueError('두 데이터셋 모두 비어 있지 않은 학습 텍스트가 필요합니다.')
        tmp.replace(corpus_path)
    finally:
        tmp.unlink(missing_ok=True)
    print(f'pretrain: {n_pretrain:,}줄', flush=True)
    return n_pair + n_pretrain


jsonl = ensure_input(PAIR_SOURCE)
pretrain_json = ensure_input(PRETRAIN_SOURCE)

%time n = extract_training_corpus(jsonl, pretrain_json, CORPUS)
print(f'합친 코퍼스 {n:,}줄, {CORPUS.stat().st_size / 1e6:.0f} MB')

with CORPUS.open(encoding='utf-8') as f:
    for _, line in zip(range(5), f):
        print(' |', line.strip()[:80])


In [ ]:
import os
# 2) SentencePiece Unigram 학습 (수 분 소요, 로그가 아래에 출력된다)
#    num_threads 기본값 = os.cpu_count() -> EM 학습 단계가 모든 코어를 사용한다
%time model_path = train(CORPUS, vocab_size=32768, num_threads=os.cpu_count())
print('저장:', model_path)
for filename in ('spm.model', 'spm.vocab'):
    shutil.copy2(OUT_DIR / filename, OUTPUT_DIR / filename)
print('Drive 저장 완료:', OUTPUT_DIR)


In [ ]:
# 3) 로드 + 기본 정보
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(MODEL_PATH))

print('vocab size :', sp.get_piece_size())
print('pad/bos/eos/unk id:', sp.pad_id(), sp.bos_id(), sp.eos_id(), sp.unk_id())
for tok in SPECIAL_TURN_TOKENS:
    print(f'{tok:>16} -> id {sp.piece_to_id(tok)}')
print('\n앞 20개 piece:', [sp.id_to_piece(i) for i in range(20)])

In [ ]:
# 4) 인코딩/디코딩 round-trip 확인
samples = [
    '안녕하세요, 국민건강보험법 제5조를 요약해 주세요.',
    '혈압이 140/90 mmHg 이상이면 고혈압입니다.',
    'The quick brown fox jumps over 13 lazy dogs.',
    '이모지 😊 와 한자 漢字, 특수문자 ㈜·※ 도 깨지지 않아야 한다.',
]
for text in samples:
    ids = sp.encode(text)
    decoded = sp.decode(ids)
    ok = decoded == unicodedata.normalize('NFKC', text)   # NFKC 정규화 후 일치해야 정상
    print(f"[{'OK ' if ok else 'DIFF'}] {len(ids):3d} tokens | {text}")
    print('      pieces:', sp.encode(text, out_type=str))
    if not ok:
        print('      decoded:', decoded)

In [ ]:
# 5) 챗 템플릿 인코딩 확인 - 학습 시 실제로 쓰는 형태 (mask=1 구간만 손실 계산)

ids, mask = encode_sample(sp, '감기에 걸렸을 때 어떻게 해야 하나요?', '충분한 휴식과 수분 섭취가 중요합니다.')
df = pd.DataFrame({'id': ids, 'piece': [sp.id_to_piece(i) for i in ids], 'loss_mask': mask})
print(f'총 {len(ids)} tokens, 손실 계산 대상 {sum(mask)} tokens')
df.T

In [ ]:
# 6) 대화 데이터(pair_train.jsonl) 토큰 통계 - 파일 전체에서 등간격 표본 4,000개
#    주의: pair_train.jsonl 은 소스 파일명 순으로 이어 붙인 것(셔플 안 됨)이라 "앞 N줄"만 보면
#    한 소스만 보게 되므로 전체 파일에서 표본을 추출한다.
jsonl = ensure_input(PAIR_SOURCE)
N_SAMPLE = 4000
n_lines = sum(1 for _ in jsonl.open(encoding='utf-8'))
stride = max(n_lines // N_SAMPLE, 1)

rows = []
with jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % stride:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        ids, mask = encode_sample(sp, obj.get('user', ''), obj.get('assistant', ''))
        rows.append({'tokens': len(ids), 'prompt': len(ids) - sum(mask), 'answer': sum(mask),
                     'chars': len(obj.get('user', '')) + len(obj.get('assistant', ''))})
stats = pd.DataFrame(rows)
stats['chars_per_token'] = stats['chars'] / stats['tokens']
print(f'{n_lines:,}줄 중 {len(stats):,}개 표본 (stride {stride})')
print(stats.describe(percentiles=[.5, .9, .95, .99]).round(1))

for L in (512, 1024, 2048):
    print(f'seq_len {L:5d} 이내 샘플: {(stats.tokens <= L).mean() * 100:5.1f}%')
print(f"\n=> 학습 seq_len 2048: p99 {stats.tokens.quantile(.99):.0f} tokens, 초과 {(stats.tokens > 2048).mean() * 100:.2f}%"
      f" | 답변 p90 {stats.answer.quantile(.9):.0f} tokens (추론 max_new_tokens 512 이상 권장)"
      f" | 평균 압축률 {stats['chars_per_token'].mean():.2f} chars/token")

In [ ]:
# 7) vocab 살펴보기 - 어떤 조각들이 학습됐는지
pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]

longest = sorted(pieces, key=len, reverse=True)[:20]
print('가장 긴 piece 20개:')
for piece in longest:
    print('  ', piece)

n_byte = sum(piece.startswith('<0x') for piece in pieces)
print(f'\nbyte-fallback piece: {n_byte}개 (256개면 정상)')
print('한국어 piece 예시:', [p for p in pieces[100:3000] if any("가" <= c <= "힣" for c in p)][:30])